### Below are imp to note \
1. while reading "load" is used only with paths (s3,adfs,azure) not with tables (specially with UNity catlaog ones) , simple use spark.read.table()
2. If you use only one DF for all notebook cells then you have to run from start in case of error Vs if you save it to another that becomes your save point to run from there 
3. Dict return None is item using get is not founf in dict.

#### List volumes 

####  Catalogue 
            ---- DATABASE / Schema 
                  ---- Tables 
                  ---- Volumes 

In [0]:
%sql 
show volumes in workspace.default

In [0]:
%sql
create  volume workspace.default.misc_data;

In [0]:
%sql 
show volumes 


#### Lets download file from Drive to Volume 
- then write it as parquet in 6 partition in same volume

In [0]:
import requests
drive_file_url = "https://drive.google.com/file/d/1tekc6cSe7xE6eIh6jYR7cAhMHAMOnuGe/view?usp=share_link"
volume_path = "/Volumes/workspace/default/misc_data/weather.csv"


r = requests.get(drive_file_url)

with open("/dbfs" + volume_path, "wb") as f:
    f.write(r.content)


In [0]:
weather_data = spark.read.format("csv").option("header",True).load("/Volumes/workspace/default/misc_data/weather_data.csv")


In [0]:
weather_data.show(5)

In [0]:
## Write csv in form of parquest. 
weather_data.repartition(6) \
    .write \
    .format('parquet') \
    .save("/Volumes/workspace/default/misc_data/weather_parq/")

In [0]:
%fs
ls /Volumes/workspace/default/misc_data/weather_parq

#### Writing Dataframe as Table 
- writing dataframe as table is possible in DBR
- common syntax - "df.write.format('delta').saveAsTable(catalog.schema.db.table_name)
- another way to register DF as table ---> df.createOrReplaceTempView() / globalView()

In [0]:
### Read Parquet ---> convert it to Table 

parq_df = spark.read \
    .format('parquet') \
    .load("/Volumes/workspace/default/misc_data/weather_parq")

display(type(parq_df))

parq_df.write.saveAsTable("workspace.default.weather_data")

#### All About Dbutils utility 
- this is used to interact with File system 
- all iunix operation like mkdir , ls, mv,cp can be done using dbutils
dbutils is a Databricks utility library used for:
1. File operations (fs)
2. Secret management (secrets)
3. Notebook parameterization (widgets)
4. Notebook orchestration (notebook)
5. Workflow communication (jobs)

In [0]:

display(dbutils.fs.ls("/Volumes/workspace/default/misc_data/"))

dbutils.fs.mkdirs('/Volumes/workspace/default/misc_data/demo_dir')

display(dbutils.fs.ls("/Volumes/workspace/default/misc_data/"))

dbutils.fs.mv("/Volumes/workspace/default/misc_data/weather_data.csv", "/Volumes/workspace/default/misc_data/demo_dir/")

# ls "Volumes/workspace/default/misc_data/demo_dir/"

In [0]:
%fs
ls "/Volumes/workspace/default/misc_data/demo_dir/"

#### Read from your S3 bucket
- remember here that we have to create External location pointing to S3 bucket 
- - how this is done is from catalog we create new xternal location , then we get Hash key that we pate in s3 bucket of aws which is opened from DBR console itself:)
- then we can access s3 bucket back in DBR Space

In [0]:
from pyspark.sql.types import StructType , StructField
from pyspark.sql.types import IntegerType, StringType, TimestampType



schema = StructType ([
  StructField("customer_id", IntegerType()),
  StructField("phone", IntegerType()),
  StructField("loaded_at", TimestampType())

])


csv_df = spark.read.option("Header", True).schema(schema).csv("s3://bucket-customer-hamonised-data/processed/cust_test.csv")

csv_df.write \
  .format('parquet') \
  .mode("overwrite") \
  .save("Volumes/ecommerce/misc/ex_vol_dummy_data")






#### External volumes VS managed volumes Vs Mountpoint


-- Exyternal volumes 
-- Volumes are ONLY READ ONLY \
-- in this case we have cloud location (s3 / azure bucket) for which \
-- 1. We first create external Location \
-- 2. Thne we create ex. volume using step 1 location \
-- 3. these vol are created under Catlagog > Schema / DB > volumes \ 
-- 4 we cant write data directly to these location , if we want to write then we have to give s3://<path> of same volume as shown below. 

#### Why we create xternal volumes when we have can use external locations ? \
1 . it gives tree linke mount point so bit userfirndly \
2.  we can simple read using volumes / bla / bla rather than typing entire externa location path everytime

#### IF U WANT TO WRITE TO VOLUMES THEN CREATE MOUNTPOINT :)
- it support writing too unlike extenal / managed vols


In [0]:

csv_df = spark.read.option("Header", True).schema(schema).csv("s3://bucket-customer-hamonised-data/processed/cust_test.csv")

# try to write to vol 
"""
FAILS 
csv_df.write \
  .format('parquet') \
  .mode("overwrite") \
  .save("Volumes/ecommerce/misc/ex_vol_dummy_data")

"""
#-- SUCCess 
csv_df.write \
  .format('parquet') \
  .mode("overwrite") \
  .save("s3://dummy-dbr-data/garbage")
# this can be seen on LHS tree view :)

### MOUNTPOINT





In [0]:
dbutils.fs.mount("s3a://dummy-dbr-data/garbage", mount_point = "/mnt/mydata")